In [ ]:
import os
import re
import sys
import math
import numpy as np
from numpy.linalg import inv
from scipy.signal import lsim
from pathlib import Path
from tqdm.notebook import tqdm

if '..' not in sys.path:
    sys.path = ['..'] + sys.path
from filter_OU_inputs import fit_TF, filter_inputs
import multitone
from multitone import newman_phase, compute_fourier_coeffs

In [ ]:
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
fontsize = 9
FIGURES_DIR = Path('figures')

In [ ]:
def rad2deg(x_rad, ref_x_deg):
    ref_y = np.asarray(ref_x_deg)
    y = np.rad2deg(np.asarray(x_rad))
    if np.any(y - ref_y > 180):
        return y - 360
    if np.any(y - ref_y < -180):
        return y + 360
    return y

In [ ]:
def find_full_var_names(all_var_names, device_names, device_types, var_names):
    full_var_names = []
    for dev_name, dev_type in zip(device_names, device_types):
        for var_name in var_names:
            full_var_names += [
                name for name in all_var_names if
                    re.match(f'.*-{dev_name}\\.{dev_type}\\.{var_name}$', name) is not None
            ]
    return full_var_names

In [ ]:
# suffix = '_lowLV'
# Ta = 2
suffix = ''
input_name = 'Genstat_01'
input_name = 'EqX_CA4CDI1501TRV_____LOAD____'
Ta = 4
data_dir = Path(f'../data/Sardinia/default_config/GFM_01{suffix}/{input_name}/Ta_{Ta}')
assert data_dir.exists(), f'{data_dir}: no such directory.'

Load the AC data: the data file contains the transfer functions computed by `compute_spectra.py`.

In [ ]:
fmin, fmax = -3, 1
points_per_decade = 50
AC_data_file = (
    data_dir / 
    'Sardegna_2021_06_03cr_1GFM{}_AC_TF_{:.1f}_{:.1f}_{}.npz'.format(
        suffix, fmin, fmax, points_per_decade
    )
)
assert AC_data_file.exists(), f'{AC_data_file}: no such file'
AC_data = np.load(AC_data_file, allow_pickle=True)
frequencies = AC_data['F']
all_var_names = AC_data['var_names'].tolist()

In [ ]:
compare_with_transient_sim = True

Load the transient data: the data file contains the results of the simulations performed with PowerFactory.

In [ ]:
if compare_with_transient_sim:
    # N_tones = 80
    N_tones = 20
    # N_tones = 1
    # N_tones = None
    F0 = 0.005
    T0 = 1 / F0
    dP = 0.01
    tran_data_file = data_dir / (
        'Sardegna_2021_06_03cr_1GFM' + suffix +
        '_tran{}_F0_{}_Hz{}.npz'.format(
            f'_multitone_N_{N_tones}' if N_tones is not None else '',
            F0,
            f'_dP_{dP}' if dP is not None else ''
        )
    )
    assert tran_data_file.exists(), f'{tran_data_file}: no such file'
    blob = np.load(tran_data_file, allow_pickle=True)
    time_tran = blob['time'].astype(float)
    tran_data = blob['data'].item()
    tran_config = blob['config'].item()
    input_waveform = tran_config['inputs'][0]['waveform']
    print('Input waveform:', input_waveform)
    if N_tones is not None:
        input_fun = eval(input_waveform)
    else:
        input_fun = np.vectorize(eval(input_waveform), otypes=[list])
    substr = re.search('\\[.*\\(', input_waveform).group()[1:]
    tran_coeff = 1
    for pos in re.finditer('[+-]', substr):
        tran_coeff = float(substr[:pos.span()[0]])
        substr = substr[pos.span()[-1]:]
    substr = substr.strip()
    for pos in re.finditer('[+*/\\-]', substr):
        stop = pos.span()[1] - 1
    print('input_coeff = 1 / ({})'.format(substr[:stop]))
    input_coeff = 1 / eval(substr[:stop])
    print('Input coefficient:', input_coeff)
    print('Tran coefficient:', tran_coeff)
    device_names = blob['device_names'].item()
    device_types_dict = {'genstat': 'ElmGenstat', 'gen': 'ElmSym', 'bus': 'ElmTerm'}
    var_names_dict = {'genstat': 's:xspeed', 'gen': 's:speed', 'bus': 'm:fe'}
    tran_data_dict = {'genstat': 's:xspeed', 'gen': 's:xspeed', 'bus': 'm:fe'}
    if False:
        try:
            u_PF = np.loadtxt(data_dir / 'delta_pref_01.txt', skiprows=1)
        except:
            u_PF = None
    else:
        u_PF = None

##### Define the inputs and outputs among the variables available in the data file:

In [ ]:
if compare_with_transient_sim:
    devices_to_consider = 'genstat', 'gen'
else:
    vars_to_consider = {
        'ElmGenstat': {
            'var_types': ['xspeed'],
            'names': ['.*Genstat_01.*']
        },
        'ElmSym': {
            'var_types': ['speed'],
            'names': ['.*_GEN_.*']
        }
    }

Inputs:

In [ ]:
input_names = [input_name]
all_input_names = AC_data['input_names'].tolist()
assert all(name in all_input_names for name in input_names), "At least one input name missing from data file"
input_idx = [all_input_names.index(name) for name in input_names]
N_inputs = len(input_idx)
print('Number of inputs:', N_inputs)
print('Input index:', *input_idx)

Outputs:

In [ ]:
full_var_names = []
if compare_with_transient_sim:
    max_N_vars = 100
    Y_tran = []
    for dev in devices_to_consider:
        tmp = find_full_var_names(
            all_var_names,
            device_names[dev][:max_N_vars],
            [device_types_dict[dev] for _ in range(max_N_vars)],
            [var_names_dict[dev].split(':')[1]]
        )
        full_var_names += tmp
        Y_tran.append(np.atleast_2d(tran_data[dev][tran_data_dict[dev]].T)[:max_N_vars])
    Y_tran = tran_coeff * np.vstack(Y_tran)
else:
    for dev_type, D in vars_to_consider.items():
        for var_type in D['var_types']:
            for dev_name in D['names']:
                pattern = dev_name + '.' + dev_type + '.' + var_type
                tmp = [var_name for var_name in all_var_names if re.match(pattern, var_name)]
                full_var_names += tmp

output_idx = [all_var_names.index(name) for name in full_var_names]
N_outputs = len(output_idx)
print('Number of outputs:', N_outputs)

Get the appropriate transfer functions:

In [ ]:
J, K = np.meshgrid(input_idx, output_idx, indexing='ij')
TF = AC_data['TF'][:, J, K]

Fit the tranfer functions using the vector fitting algorithm and create the `lsim` objects:

In [ ]:
systems, _ = fit_TF(frequencies, TF)

Define the inputs:

In [ ]:
if compare_with_transient_sim:
    dt = tran_config['dt']
    tend = tran_config['tstop']
    tmin = 100
else:
    F0 = 10e-3
    T0 = 1 / F0
    N_tones = 10
    dt = 1 / 100
    tend = 12 * T0
    tmin = 2 * T0

time = np.r_[0 : tend : dt]

if compare_with_transient_sim:
    u = input_fun(time)
    if len(u) == 2:
        u = u[0]
    if N_tones is None:
        u = np.array([_[0] for _ in u])
else:
    input_coeff = 1
    u = input_coeff * multitone.multitone(time, N_tones, omega0=2 * math.pi * F0)
    
U = np.tile(
    u,
    (N_inputs, 1)
)
N_samples = time.size

Filter the inputs using the `lsim` objects above:

In [ ]:
Y = filter_inputs(systems, U, time)

Plot the results:

In [ ]:
T0 = 1 / F0
N_T = min(10, (tend - tmin) // T0)
tstart = tend - N_T * T0
print(f'Removing the first {tstart} s.')

N_traces = 1

fig, ax = plt.subplots(2, 1, figsize=(6,4), height_ratios=(1,3), sharex=True)
ax[0].plot(time, U.T, 'k', lw=1)

cmap = ['tab:red', 'tab:green']

K = 1
for j in range(N_traces):
    i = j + K
    lbl = full_var_names[i].split('-')[-1]
    if compare_with_transient_sim:
        idx = time_tran > tstart
        ax[1].plot(time_tran[idx], (Y_tran[i, idx] - Y_tran[i, idx].mean()),
                   color=cmap[j], lw=1, label=lbl + ' (PF tran)')
    idx = time > tstart
    ax[1].plot(time[idx], Y[i, idx] - Y[i, idx].mean(), color=cmap[j+1], lw=1, label=lbl + ' (ss)')
ax[0].set_ylabel('U(t)')
ax[1].set_ylabel('Y(t)')
ax[1].legend(loc='lower left', frameon=True, fontsize=7)
ax[1].set_xlabel('Time (s)')
ax[1].set_xlim([tstart, tend])
sns.despine()
fig.tight_layout()
outfile = tran_data_file.parent / (tran_data_file.stem + '.pdf')
plt.savefig(outfile)

In [ ]:
if N_tones is not None:
    freqs = np.array([(i + 1) * F0 for i in range(N_tones)])
    phases = newman_phase(np.arange(N_tones) + 1, N_tones)
    A = np.sqrt(N_tones / 2)
else:
    freqs = np.array([F0])
    phases = np.array([0.0])
    A = np.array([1.0])

if len(freqs) > 1:
    unwrap = lambda x: np.unwrap(x, axis=0)
else:
    unwrap = lambda x: x

idx = time >= tstart
ΔY = (Y[:, idx] - Y[:, idx].mean(axis=1)[:, np.newaxis]).T
coeffs = compute_fourier_coeffs(
    freqs,
    time[idx] - tstart,
    ΔY,
    phases,
    A
)

C = input_coeff * np.abs(coeffs)
ϕ = unwrap(np.angle(coeffs))

if compare_with_transient_sim:
    idx = time_tran >= tstart
    ΔY_tran = (Y_tran[:, idx] - Y_tran[:, idx].mean(axis=1)[:, np.newaxis]).T
    coeffs_tran = compute_fourier_coeffs(
        freqs,
        time_tran[idx] - tstart,
        ΔY_tran,
        phases,
        A
    )
    C_tran = input_coeff * np.abs(coeffs_tran)
    ϕ_tran = unwrap(np.angle(coeffs_tran))

In [ ]:
n = 0
F_target = (n + 1) * F0
try:
    i = np.where(frequencies == F_target)[0][0]
except:
    i = np.argmin(np.abs(frequencies - F_target))
    print('Frequency {} Hz not available: closest one is {:g} Hz (discrepancy {:.2f}%).'.\
        format(F_target, frequencies[i], (F_target - frequencies[i]) / F_target * 100))
j = 0
C_TF, ϕ_TF = np.abs(TF[i, j, K]), np.angle(TF[i, j, K])

phi_TF = np.rad2deg(ϕ_TF)
phi = rad2deg(ϕ[n, K], phi_TF)
phi_tran = rad2deg(ϕ_tran[n, K], phi_TF)

print('Transfer function from PF modal analysis:')
print('    abs = {:g}'.format(C_TF))
print('  phase = {:g} deg'.format(phi_TF))
print('Fourier coefficient from small-signal simulation:')
print('    abs = {:g}'.format(C[n, K]))
print('  phase = {:g} deg'.format(phi))
print('Fourier coefficient from PF transient simulation:')
print('    abs = {:g}'.format(C_tran[n, K]))
print('  phase = {:g} deg'.format(phi_tran))

In [ ]:
if N_tones is not None:
    idx = (frequencies > 2/3 * F0) & (frequencies < 1.5 * N_tones * F0)
else:
    idx = (frequencies > 2/3 * F0) & (frequencies < 1.5 * F0)
C_TF = np.abs(TF[idx, j, K])
phi_TF = np.rad2deg(np.unwrap(np.angle(TF[idx, j, K])))
if N_tones is None or N_tones == 1:
    phi = rad2deg(ϕ[:, K], phi_TF)
    phi_tran = rad2deg(ϕ_tran[:, K], phi_TF)
else:
    # phi = np.unwrap(np.rad2deg(ϕ[:, K]))
    # phi_tran = np.unwrap(np.rad2deg(ϕ_tran[:, K]))
    phi = np.rad2deg(ϕ[:, K])
    phi_tran = np.rad2deg(ϕ_tran[:, K])

ms = 5
ew = 1
coeff = 1e3
fig, ax = plt.subplots(2, 1, figsize=(6,4), sharex=True)
ax[0].plot(frequencies[idx] * coeff, C_TF, 'k', lw=1, label='TF')
ax[0].plot(freqs * coeff, C[:, K], 'o', color='tab:green', markersize=ms,
           markeredgewidth=ew, markerfacecolor='w', label='ss')
ax[0].plot(freqs * coeff, C_tran[:, K], 'o', color='tab:red', markersize=ms,
           markeredgewidth=ew, markerfacecolor='w', label='PF tran')
ax[1].plot(frequencies[idx] * coeff, phi_TF, 'k', lw=1)
ax[1].plot(freqs * coeff, phi, 'o', color='tab:green', markersize=ms,
           markeredgewidth=ew, markerfacecolor='w')
ax[1].plot(freqs * coeff, phi_tran, 'o', color='tab:red', markersize=ms,
           markeredgewidth=ew, markerfacecolor='w')
ax[0].set_xscale('log')
ax[0].legend(loc='upper left', frameon=False, fontsize=8)
ax[0].set_ylabel(r'$|\hat{X}(j\omega)|$')
ax[1].set_ylabel(r'$\phi(X(j\omega))$')
ax[1].set_xlabel('Frequency [{}Hz]'.format('m' if coeff == 1e3 else ''))
sns.despine()
fig.tight_layout()
outfile = tran_data_file.parent / (tran_data_file.stem + '_TF.pdf')
plt.savefig(outfile)